# Balsara 1 shock tube test

This notebook runs a shock-tube test with AsterX, and visualizes the data saved in TSV and OpenPMD format. 


# Test setup
This is a generic shock-tube test with magnetic fields, where the shock propagates along x-direction. The initial conditions are the same as of the Balsara 1 shock tube problem. Here, the right and left states along x are initalized to different values as follows:

- Left state is initialized as ($\rho, v^x, v^y, v^z, P$) = $[1,0,0,1]$, and the right state as $[0.125,0,0,0.1]$. 

- For the magnetic field components, left state is ($B^x, B^y, B^z$) = $[0.5,1.0,0.0]$, and the right state $[0.5,-1.0,0.0]$.

- Here, we use the ideal gas EOS with $\gamma = 2$. 

- Initial data is set using the thorn ```AsterSeeds```.

- Grid domain is $[-0.5,+0.5]$ along x, with 200 grid-cells.

- `Linear extrapolation` is used as boundary conditions in all directions.

This gives the initial contact discontinuity at $x=0.0$.


# Reconstruction Method

This test will use the so-called WENO-Z reconstruction method. It is a fifth-order reconstruction method designed for highly accurate evolutions of smooth flows but also the stable evolution of sharp features such as shocks.

A reference on the method is given by [Borges et al., 2008](https://ui.adsabs.harvard.edu/link_gateway/2008JCoPh.227.3191B/doi:10.1016/j.jcp.2007.11.038).

## Reconstruction: the WENO-Z scheme (5th order)

The minmod limiter is robust but only **second-order**, and it degrades to first
order at smooth extrema (the slope is clipped to zero). For problems with both
sharp features *and* delicate smooth structure — turbulence, MHD instabilities,
oscillating neutron stars — we want **high-order** accuracy in smooth regions
while still avoiding oscillations at shocks. That is what **WENO** (Weighted
Essentially Non-Oscillatory) schemes deliver, and AsterX uses the improved
**WENO-Z** variant mentioned above.

> **Convention used throughout.** Every cell $I$ has a **minus (left) face** at
> $x_{I-1/2}$ and a **plus (right) face** at $x_{I+1/2}$. We write the
> reconstructed values there as $q^{-}_{I}$ and $q^{+}_{I}$, matching the code's
> `var_m` and `var_p`. *(Beware: the Riemann-solver literature often writes
> $u^{\mp}_{I+1/2}$ for the states on either side of a single interface — there
> the "minus" state is the value coming from the cell on the left, i.e. our
> $q^{+}$ of that left cell.)*

### Candidate stencils

To reconstruct the **plus (right) face** value $q^{+}_{I}$ of a central cell
$I$, WENO5 uses a five-point stencil $\{I^{--}, I^{-}, I, I^{+}, I^{++}\}$ split
into **three overlapping 3-point substencils**:

$$
S_0=\{I^{--},I^{-},I\},\qquad
S_1=\{I^{-},I,I^{+}\},\qquad
S_2=\{I,I^{+},I^{++}\}.
$$

Each $S_k$ gives its own **third-order** estimate of $q^{+}_{I}$:

$$
p_0=\tfrac{1}{6}\big(2q_{I^{--}}-7q_{I^{-}}+11q_{I}\big),\quad
p_1=\tfrac{1}{6}\big(-q_{I^{-}}+5q_{I}+2q_{I^{+}}\big),\quad
p_2=\tfrac{1}{6}\big(2q_{I}+5q_{I^{+}}-q_{I^{++}}\big).
$$

Combined with the **optimal linear weights** $d=(\tfrac{1}{10},\tfrac{6}{10},
\tfrac{3}{10})$ they reproduce the full 5th-order reconstruction of $q^{+}_{I}$ —
*but only where the data is smooth*. The **minus (left) face** value $q^{-}_{I}$
is built from the same stencil with the mirror-image polynomials and the
mirrored weights $d=(\tfrac{3}{10},\tfrac{6}{10},\tfrac{1}{10})$.

### Smoothness indicators

Each substencil is assigned a smoothness indicator $\beta_k$ (large = oscillatory):

$$
\begin{aligned}
\beta_0 &= \tfrac{13}{12}\big(q_{I^{--}}-2q_{I^{-}}+q_{I}\big)^2
        + \tfrac{1}{4}\big(q_{I^{--}}-4q_{I^{-}}+3q_{I}\big)^2,\\
\beta_1 &= \tfrac{13}{12}\big(q_{I^{-}}-2q_{I}+q_{I^{+}}\big)^2
        + \tfrac{1}{4}\big(q_{I^{-}}-q_{I^{+}}\big)^2,\\
\beta_2 &= \tfrac{13}{12}\big(q_{I}-2q_{I^{+}}+q_{I^{++}}\big)^2
        + \tfrac{1}{4}\big(3q_{I}-4q_{I^{+}}+q_{I^{++}}\big)^2.
\end{aligned}
$$

### The WENO-Z weights

Classical WENO-JS ([Jiang and Shu, 1996](https://www.sciencedirect.com/science/article/abs/pii/S0021999196901308)) forms its weights as
$\alpha_k = d_k/(\beta_k+\varepsilon)^2$, which is overly dissipative and loses
accuracy at critical points. **WENO-Z** introduces a *global* smoothness measure
built from the **whole** stencil,

$$
\tau_5 = \big|\beta_0 - \beta_2\big|,
$$

and defines

$$
\alpha_k = d_k\left(1 + \frac{\tau_5}{\beta_k+\varepsilon}\right),
\qquad
\omega_k = \frac{\alpha_k}{\alpha_0+\alpha_1+\alpha_2}.
$$

Here $\varepsilon$ is a tiny floor (`weno_eps`) that prevents division by zero.
In smooth regions $\tau_5 \ll \beta_k$, so $\omega_k \to d_k$ and the scheme
recovers full 5th-order accuracy — *including at smooth extrema*, where minmod
would have dropped to first order. Near a discontinuity, the offending
substencil has a large $\beta_k$, its $\omega_k\to 0$, and that stencil is
"switched off" — the essentially non-oscillatory property.

### Final reconstruction

The **plus (right) face** value is the convex combination of the three
candidates,

$$
q^{+}_{I} \;=\; \sum_{k=0}^{2}\omega_k\,p_k ,
$$

which is the code's `var_p`. The **minus (left) face** value $q^{-}_{I}$
(`var_m`) is obtained identically from the mirror-image polynomials and the
mirrored optimal weights. To get the two states meeting at a *single* interface,
`wenoz_reconstruct` applies this to the two cells sharing the face and keeps the
**plus-face** value of the left cell ($q^{L}$) and the **minus-face** value of
the right cell ($q^{R}$).

**Map to the code:** `betaZ` $=\beta_k$; `tau5` $=\tau_5$; `aux_alphaZ`
$=1+\tau_5/(\beta_k+\varepsilon)$; `wt` $=d_k$; `omegaZ` $=\omega_k$;
`var_m`/`var_p` are the minus/plus face values $q^{-}_{I}/q^{+}_{I}$. The linear
weights are mirrored between the two faces exactly as in the
`alphaZ(...)( 0 | 1 )` assignments.

In [ ]:
"""
Demo of the WENO-Z (5th order) reconstruction used in ReconX, with the
second-order minmod reconstruction overlaid for comparison.

This cell reproduces ReconX::wenoz / ReconX::wenoz_reconstruct
(and ReconX::minmod) in NumPy on a smooth Gaussian bump, resolved by only a
handful of cells so the difference between the schemes is easy to see.
"""

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# Faithful NumPy port of ReconX::minmod (for comparison)
# --------------------------------------------------------------------------
def minmod(x, y):
    if np.sign(x) != np.sign(y):
        return 0.0
    return x if abs(x) < abs(y) else y

# --------------------------------------------------------------------------
# Faithful NumPy port of ReconX::wenoz / ReconX::wenoz_reconstruct
# --------------------------------------------------------------------------
def wenoz(Imm, Im, I, Ip, Ipp, eps=1.0e-26):
    """5th-order WENO-Z reconstruction of cell I's two interface values.
    Returns (var_m, var_p): the minus (left) and plus (right) face values."""
    # smoothness indicators (Borges et al. 2008)
    b0 = (13.0/12.0)*(Imm - 2.0*Im + I)**2 + 0.25*(Imm - 4.0*Im + 3.0*I)**2
    b1 = (13.0/12.0)*(Im  - 2.0*I  + Ip)**2 + 0.25*(Im - Ip)**2
    b2 = (13.0/12.0)*(I   - 2.0*Ip + Ipp)**2 + 0.25*(3.0*I - 4.0*Ip + Ipp)**2

    # global smoothness (Borges eq. 25) and unnormalized Z-factors
    tau5 = abs(b0 - b2)
    a0 = 1.0 + tau5/(b0 + eps)
    a1 = 1.0 + tau5/(b1 + eps)
    a2 = 1.0 + tau5/(b2 + eps)

    # optimal linear weights; mirrored per side
    d0, d1, d2 = 3.0/10.0, 3.0/5.0, 1.0/10.0

    # minus side
    am0, am1, am2 = d0*a0, d1*a1, d2*a2
    sm = am0 + am1 + am2
    wm0, wm1, wm2 = am0/sm, am1/sm, am2/sm

    # plus side (optimal weights mirrored)
    ap0, ap1, ap2 = d2*a0, d1*a1, d0*a2
    sp = ap0 + ap1 + ap2
    wp0, wp1, wp2 = ap0/sp, ap1/sp, ap2/sp

    # reconstruction polynomials
    var_m = ( (wm2/6.0)*(2.0*Ipp - 7.0*Ip + 11.0*I)
            + (wm1/6.0)*(-1.0*Ip + 5.0*I + 2.0*Im)
            + (wm0/6.0)*(2.0*I + 5.0*Im - 1.0*Imm) )
    var_p = ( (wp0/6.0)*(2.0*Imm - 7.0*Im + 11.0*I)
            + (wp1/6.0)*(-1.0*Im + 5.0*I + 2.0*Ip)
            + (wp2/6.0)*(2.0*I + 5.0*Ip - 1.0*Ipp) )
    return var_m, var_p

def wenoz_reconstruct(Immm, Imm, Im, Ip, Ipp, Ippp, eps=1.0e-26):
    """Returns (qL, qR) at the face between cells Im and Ip
    (matches ReconX::wenoz_reconstruct)."""
    rc_Im = wenoz(Immm, Imm, Im, Ip, Ipp, eps)   # central cell = Im
    rc_Ip = wenoz(Imm, Im, Ip, Ipp, Ippp, eps)   # central cell = Ip
    return rc_Im[1], rc_Ip[0]

# --------------------------------------------------------------------------
# Test profile: a smooth Gaussian bump on [0.2, 0.4], few cells
# --------------------------------------------------------------------------
N  = 16
x  = np.linspace(0.2, 0.4, N)          # cell centres (uniform grid)
dx = x[1] - x[0]
q  = np.exp(-((x - 0.30) / 0.06)**2)   # smooth Gaussian bump

# --------------------------------------------------------------------------
# Per-cell reconstructed interface values
#   WENO-Z needs cells i-2 .. i+2 ; minmod needs i-1 .. i+1
# --------------------------------------------------------------------------
xc, wz_m, wz_p, mm_m, mm_p = [], [], [], [], []
for i in range(2, N - 2):
    vm, vp = wenoz(q[i-2], q[i-1], q[i], q[i+1], q[i+2])
    s = minmod(q[i+1] - q[i], q[i] - q[i-1])   # minmod limited slope of cell i
    xc.append(x[i])
    wz_m.append(vm); wz_p.append(vp)
    mm_m.append(q[i] - 0.5*s); mm_p.append(q[i] + 0.5*s)
xc, wz_m, wz_p, mm_m, mm_p = map(np.array, (xc, wz_m, wz_p, mm_m, mm_p))

# WENO-Z left/right states at each interior face (qL = right face of left cell,
# qR = left face of right cell) -- equivalently wenoz_reconstruct()
xf, qL, qR = [], [], []
for i in range(2, N - 3):
    L, R = wenoz_reconstruct(q[i-2], q[i-1], q[i], q[i+1], q[i+2], q[i+3])
    xf.append(0.5*(x[i] + x[i+1]))
    qL.append(L); qR.append(R)
xf, qL, qR = map(np.array, (xf, qL, qR))

# --------------------------------------------------------------------------
# Plot
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=[7.0, 4.0])

# exact profile (reference "truth")
xx = np.linspace(0.2, 0.4, 400)
ax.plot(xx, np.exp(-((xx - 0.30)/0.06)**2), "-", color="black", lw=1.0,
        alpha=0.6, label="exact")

# cell averages
ax.plot(x, q, "o", color="0.4", ms=4, label="cell average $q_i$")

# minmod (2nd order) per-cell reconstruction
for k, i in enumerate(range(2, N - 2)):
    ax.plot([x[i] - dx/2, x[i] + dx/2], [mm_m[k], mm_p[k]],
            color="seagreen", lw=1.0, ls="--", zorder=2,
            label="minmod (2nd order)" if k == 0 else None)

# WENO-Z (5th order) per-cell reconstruction (segment joins the two
# reconstructed interface values -- a guide to the eye)
for k, i in enumerate(range(2, N - 2)):
    ax.plot([x[i] - dx/2, x[i] + dx/2], [wz_m[k], wz_p[k]],
            color="steelblue", lw=1.6, zorder=3,
            label="WENO-Z (5th order)" if k == 0 else None)

# WENO-Z reconstructed face states
ax.plot(xf, qL, "<", color="crimson",     ms=6, zorder=4, label=r"$q^{L}$ (left state)")
ax.plot(xf, qR, ">", color="midnightblue", ms=6, zorder=4, label=r"$q^{R}$ (right state)")

ax.set_title("Reconstruction: minmod (2nd order) vs. WENO-Z (5th order)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("q")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 1. Steps to perform the simulation

In [ ]:
# this allows you to use "cd" in cells to change directories instead of requiring "%cd"
%automagic on
from etsetup import *

First, let's first move to the Cactus folder:

In [ ]:
cd ~/Cactus

Now, let's create the parameter file to be used for this simulation:

In [ ]:
%%writefile ./par/Balsara1_shocktube_xdir_WENOZ.par

ActiveThorns = "
    CarpetX
    IOUtil
    ODESolvers
    TimerReport
    ADMBaseX
    HydroBaseX
    TmunuBaseX
    AsterMasks
    AsterSeeds
    AsterX
    EOSX
"
 
$nlevels = 1
$ncells = 200

Cactus::cctk_show_schedule = yes

Cactus::presync_mode = "mixed-error"

Cactus::terminate = "time"
Cactus::cctk_final_time = 0.40

ADMBaseX::initial_data            = "Cartesian Minkowski"
ADMBaseX::initial_lapse           = "one"
ADMBaseX::initial_shift           = "zero"
ADMBaseX::initial_dtlapse         = "none"
ADMBaseX::initial_dtshift         = "none"

CarpetX::verbose = no

CarpetX::xmin = -0.5
CarpetX::ymin = -0.5
CarpetX::zmin = -0.5

CarpetX::xmax = +0.5
CarpetX::ymax = +0.5
CarpetX::zmax = +0.5

CarpetX::boundary_x = "linear extrapolation"
CarpetX::boundary_y = "linear extrapolation"
CarpetX::boundary_z = "linear extrapolation"

CarpetX::boundary_upper_x = "linear extrapolation"
CarpetX::boundary_upper_y = "linear extrapolation"
CarpetX::boundary_upper_z = "linear extrapolation"

CarpetX::ncells_x = $ncells
CarpetX::ncells_y = 2
CarpetX::ncells_z = 2

CarpetX::max_num_levels = $nlevels
CarpetX::regrid_every = 100000
CarpetX::blocking_factor_x = 1
CarpetX::blocking_factor_y = 1
CarpetX::blocking_factor_z = 1
CarpetX::regrid_error_threshold = 0.01

CarpetX::prolongation_type = "ddf"
CarpetX::ghost_size = 3
CarpetX::dtfac = 0.25
 
AsterSeeds::test_type = "1DTest"
AsterSeeds::test_case = "Balsara1"
AsterSeeds::shock_dir = "x"

AsterX::debug_mode = "no"
AsterX::flux_type = "HLLE"
AsterX::update_tmunu = "no"

ReconX::reconstruction_method = "wenoz"

Con2PrimFactory::c2p_prime = "Noble"
Con2PrimFactory::c2p_second = "Palenzuela"
Con2PrimFactory::c2p_tol = 1e-8
Con2PrimFactory::max_iter = 100
Con2PrimFactory::rho_abs_min = 1e-8
Con2PrimFactory::unit_test = "no"
Con2PrimFactory::B_lim = 1e8
Con2PrimFactory::vw_lim = 1e8
Con2PrimFactory::Ye_lenient = "yes"

EOSX::evolution_eos = "IdealGas"
EOSX::gl_gamma = 2.0
EOSX::poly_gamma = 2.0
EOSX::rho_max = 1e8
EOSX::eps_max = 1e8

ODESolvers::method = "RK4"

IO::out_dir = $parfile
IO::out_every = 10 

CarpetX::out_tsv_vars = "
    HydroBaseX::rho
    HydroBaseX::vel
    HydroBaseX::eps
    HydroBaseX::press
    HydroBaseX::Bvec
"

CarpetX::out_openpmd_vars = "
    HydroBaseX::rho
    HydroBaseX::vel
    HydroBaseX::eps
    HydroBaseX::press
    HydroBaseX::Bvec
"

TimerReport::out_every = 10
TimerReport::out_filename = "TimerReport"
TimerReport::output_all_timers_together = yes
TimerReport::output_all_timers_readable = yes
TimerReport::n_top_timers = 50

Then, submit the simulation using the following command:

In [ ]:
%%bash
rm -rf $HOME/simulations/Balsara1_shocktube_xdir_WENOZ
./simfactory/bin/sim create-submit Balsara1_shocktube_xdir_WENOZ \
  --parfile=./par/Balsara1_shocktube_xdir_WENOZ.par --procs=2 --num-threads=1 --walltime=00:20:00

The above command creates and runs the simulation ```Balsara1_shocktube_xdir_WENOZ```, using the configuration ```sim```. The data is saved in the directory ```./simulations/Balsara1_shocktube_xdir_WENOZ```.



In [ ]:
%%bash
# watch log output, following along as new output is produced
cd ~/Cactus
./simfactory/bin/sim show-output --follow Balsara1_shocktube_xdir_WENOZ

# 2. 1D Data Output -TSV Format

Norms and 1D data is saved using the TSV format which can be directly visualized using standard Python modules:

A small example toolkit for TSV data can be imported here:

  ## `carpetx_tsv_classes` — reading CarpetX `.tsv` output

  This helper module turns CarpetX `.tsv` output into NumPy arrays. It has two layers: **discovery**
  classes that scan an output folder and catalog what's available, and **loader** classes/functions
  that read a single file.

  ### Discovery — *what's in this output folder?*

  - **`OneDimTsv_OneOut(path)`** — scans `path` for 1D line-output files (`*.it*.tsv`) and parses 
  their names into variable / iteration / direction.
    - `get_vars()` → list of variables found
    - `get_iterations(var, direc)` → sorted array of iterations available for that variable and 
  direction (`'x'`, `'y'`, or `'z'`)
    - `get_path_list()` → all matching file paths

  - **`NormsTsv_OneOut(path)`** — scans `path` for **norm / time-series** files (`*.tsv` *without* an
  `.itNNNN` tag).
    - `get_vars()` → list of variables found
    - `get_path_list()` → all matching file paths

  ### Loaders — *read one file into arrays*
  
  - **`LoadTsv(path)`** — loads a single `.tsv` and exposes its columns by name.
    - `get(name)` → that column as a 1D array
    - `get_keys()` → available column names

  - **`LoadAllTimersTsv(path)`** — specialized loader for the `AllTimers.tsv` profiling file, which
  stores a different column set at `it=0` than at `it>0`. `get(name)` stitches the two together
  transparently. Also provides `get_clock()` and `get_clock_units()`. 

  ### Convenience functions — *discovery + loader in one step*

  - **`LoadNormsTsv(out, var)`** — given a `NormsTsv_OneOut` and a variable name, returns the right
  loader (`LoadAllTimersTsv` for `AllTimers`, otherwise `LoadTsv`).
  - **`LoadOneDimTsv(out, var, direction, it)`** — given a `OneDimTsv_OneOut`, returns a `LoadTsv` for
  that variable / direction / iteration.

  ### Typical usage

  ```python
  import carpetx_tsv_classes as ctx

  out  = ctx.OneDimTsv_OneOut("/path/to/simulation/output")
  print(out.get_vars())                         # which variables are available
  print(out.get_iterations("hydrobasex-rho", "x"))         # iterations along x for rho

  data = ctx.LoadOneDimTsv(out, "hydrobasex-rho", "x", 128) # load rho along x at iteration 128
  x    = data.get("x")
  rho  = data.get("rho")
  ```

  ### Norms / time-series example

  ```python
  import carpetx_tsv_classes as ctx

  norms = ctx.NormsTsv_OneOut("/path/to/simulation/output")
  print(norms.get_vars())              # which norm/time-series variables exist

  data = ctx.LoadNormsTsv(norms, "hydrobasex-rho")  # load the rho norm time series
  print(data.get_keys())               # available columns (e.g. iteration, time, ...)
  t   = data.get("time")
  rho = data.get("rho")

  # AllTimers profiling file works the same way (returns a LoadAllTimersTsv):
  timers = ctx.LoadNormsTsv(norms, "AllTimers")
  print(timers.get_clock(), timers.get_clock_units())
  ```

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/AsterX_Tutorial"))
import carpetx_tsv_classes as ctx

### Identify correct output directory:

In [ ]:
%%bash

ls /home/mchabanov/simulations/Balsara1_shocktube_xdir_MINMOD/output-0000/Balsara1_shocktube_xdir_MINMOD/hydrobasex-rho.it000000*
###
echo ''
###
ls /home/mchabanov/simulations/Balsara1_shocktube_xdir_WENOZ/output-0000/Balsara1_shocktube_xdir_WENOZ/hydrobasex-rho.it000000*

### Check available variables and iterations:

In [ ]:
out_mm  = ctx.OneDimTsv_OneOut(os.path.expanduser("~/simulations/Balsara1_shocktube_xdir_MINMOD/output-0000/Balsara1_shocktube_xdir_MINMOD"))
print(out_mm.get_vars())                                    # which variables are available
print(out_mm.get_iterations("hydrobasex-rho", "x"))         # iterations along x for rho
###
print('')
###
out_wz  = ctx.OneDimTsv_OneOut(os.path.expanduser("~/simulations/Balsara1_shocktube_xdir_WENOZ/output-0000/Balsara1_shocktube_xdir_WENOZ"))
print(out_wz.get_vars())                                    # which variables are available
print(out_wz.get_iterations("hydrobasex-rho", "x"))         # iterations along x for rho

### Load and plot variables:

In [ ]:
data_mm = ctx.LoadOneDimTsv(out_mm, "hydrobasex-rho", "x", 320) # load rho along x at iteration 320
x_mm    = data_mm.get("x")
rho_mm  = data_mm.get("rho")

data_wz = ctx.LoadOneDimTsv(out_wz, "hydrobasex-rho", "x", 320) # load rho along x at iteration 320
x_wz    = data_wz.get("x")
rho_wz  = data_wz.get("rho")

# Set up the figure (no colorbar needed for a line plot)
fig    = plt.figure(figsize=[14.0, 6.75])
axplot = fig.add_axes([0.14, 0.16, 0.80, 0.74])

# Plot a

axplot.set_title("Rest-mass density", fontsize=20., fontweight="bold", color="midnightblue")
axplot.set_xlabel("x", fontsize=14.)
axplot.set_ylabel("rho", fontsize=14.)
axplot.tick_params(labelsize=14)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))

# sort by coordinate so the line is drawn left-to-right
order_mm = np.argsort(x_mm)
axplot.plot(x_mm[order_mm], rho_mm[order_mm], color="midnightblue", lw=2.0, label="minmod")
axplot.set_ylim(0.0, 1.1)

order_wz = np.argsort(x_wz)
axplot.plot(x_wz[order_wz], rho_wz[order_wz], color="forestgreen", lw=2.0, label="weno-z")
axplot.set_ylim(0.0, 1.1)

# Print the current iteration (axes-fraction coords, so it stays put)
axplot.text(0.70, 0.88, f"Iteration {320}", transform=axplot.transAxes,
            fontsize=16., fontweight="bold", color="black")

axplot.legend(fontsize=16.)

In [ ]:
plt.close(fig)

### Full movie script:

In [ ]:
"""
Animate 1D CarpetX .tsv line output as a movie (value vs. coordinate, one
frame per iteration), using the discovery/loader classes in
carpetx_tsv_classes.
"""

from celluloid import Camera
from IPython.display import display, HTML

# --------------------------------------------------------------------------
# Discovery object (point this at your simulation output folder)
# --------------------------------------------------------------------------
out_mm = ctx.OneDimTsv_OneOut(os.path.expanduser("~/simulations/Balsara1_shocktube_xdir_MINMOD/output-0000/Balsara1_shocktube_xdir_MINMOD"))
out_wz = ctx.OneDimTsv_OneOut(os.path.expanduser("~/simulations/Balsara1_shocktube_xdir_WENOZ/output-0000/Balsara1_shocktube_xdir_WENOZ"))

# --- choose what to plot ---
var       = "hydrobasex-rho"   # a name from out.get_vars()
direction = "x"                # "x", "y", or "z"

iterations = out_mm.get_iterations(var, direction)
print(f"{len(iterations)} iterations for {var} along {direction}")

# peek at one file so we know the exact column names
sample = ctx.LoadOneDimTsv(out_mm, var, direction, iterations[0])
print("columns:", sample.get_keys())

# adjust these two if get_keys() shows different names:
coord_col = direction   # the coordinate column (usually "x"/"y"/"z")
value_col = "rho"       # the data column

# --------------------------------------------------------------------------
# Build the movie
# --------------------------------------------------------------------------
# Precompute a stable y-range so the axis doesn't jump between frames
ymin_mm, ymax_mm = np.inf, -np.inf
ymin_wz, ymax_wz = np.inf, -np.inf
for it in iterations:
    v_mm = ctx.LoadOneDimTsv(out_mm, var, direction, it).get(value_col)
    ymin_mm, ymax_mm = min(ymin_mm, v_mm.min()), max(ymax_mm, v_mm.max())
    v_wz = ctx.LoadOneDimTsv(out_wz, var, direction, it).get(value_col)
    ymin_wz, ymax_wz = min(ymin_wz, v_wz.min()), max(ymax_wz, v_wz.max())

ymin = min(ymin_mm,ymin_wz)
ymax = max(ymax_mm,ymax_wz)

# Set up the figure (no colorbar needed for a line plot)
fig    = plt.figure(figsize=[14.0, 6.75])
axplot = fig.add_axes([0.14, 0.16, 0.80, 0.74])

axplot.set_title("Rest-mass density", fontsize=20., fontweight="bold", color="midnightblue")
axplot.set_xlabel(direction, fontsize=16.)
axplot.set_ylabel(value_col, fontsize=16.)
axplot.tick_params(labelsize=14)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))

# Initialize the camera
camera = Camera(fig)

# Print frames
for it in iterations:
    data_mm  = ctx.LoadOneDimTsv(out_mm, var, direction, it)
    data_wz  = ctx.LoadOneDimTsv(out_wz, var, direction, it)
    
    coord_mm = data_mm.get(coord_col)
    value_mm = data_mm.get(value_col)

    coord_wz = data_wz.get(coord_col)
    value_wz = data_wz.get(value_col)

    # sort by coordinate so the line is drawn left-to-right
    order_mm = np.argsort(coord_mm)
    axplot.plot(coord_mm[order_mm], value_mm[order_mm], color="midnightblue", lw=2.0, label="minmod")
    axplot.set_ylim(ymin_mm, ymax_mm)

    order_wz = np.argsort(coord_wz)
    axplot.plot(coord_wz[order_wz], value_wz[order_wz], color="forestgreen", lw=2.0, label="weno-z")
    axplot.set_ylim(ymin_wz, ymax_wz)

    # Print the current iteration (axes-fraction coords, so it stays put)
    axplot.text(0.15, 0.88, f"Iteration {it}", transform=axplot.transAxes,
                fontsize=16., fontweight="bold", color="black")

    if it == 0:
        axplot.legend(fontsize=16.)

    # Snapshot for the animation
    camera.snap()

animation = camera.animate(interval=120, blit=True)

# Inline playback in the notebook. display() makes it render even when this
# isn't the last line of the cell. plt.close() stops a static duplicate frame
# from also being shown.
plt.close(fig)
display(HTML(animation.to_jshtml()))

# --------------------------------------------------------------------------
# Optional: save to a file
# --------------------------------------------------------------------------
# animation.save("rho_1d.mp4", dpi=150)          # needs ffmpeg
# animation.save("rho_1d.gif", writer="pillow")  # no ffmpeg required

## 2. Steps to visualize 2D simulation data

The 2D data can be saved in both Silo format (which can be visualised, for instance, via VisIt) and in OpenPMD format. 

For further info on Silo, please visit: https://wci.llnl.gov/simulation/computer-codes/silo

For further info about OpenPMD, please visit:

- Official website:  https://www.openpmd.org
- GitHub repository: https://github.com/openPMD
- Documentation:     https://openpmd-api.readthedocs.io

Now, let's go back to the home directory:

In [ ]:
cd ~/

Import all the required modules:

In [ ]:
%pip install --user celluloid

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from celluloid import Camera
import openpmd_api as io

Set the Matplotlib backend to `notebook`, not `inline`, since we'll want to animate some figures and the latter is not compatible with that

In [ ]:
%matplotlib inline

Open a .bp file (ADIOS2 extension) as an OpenPMD **'series'**, which is a collection of **'iterations'**, each of which contains **'records'**, which are sets of either structured data --- **'meshes'** --- or unstructured data --- **'particles'**. 

AsterX only outputs mesh data. Each record has one or more **'components'**: for example, a record representing a scalar field only has one component, while a record representing a vector field has three.

In [ ]:
home = os.environ["HOME"]
series = io.Series(os.path.join(home, "simulations",
                                "Balsara1_shocktube_xdir_WENOZ",
                                "output-0000","Balsara1_shocktube_xdir_WENOZ",
                                "Balsara1_shocktube_xdir_WENOZ.it%08T.bp5"), io.Access.read_only)

All iterations in our series have the same structure, i.e., they contain
the same records, since they all represent the same output, just at
different times. 

Here we define an empty Python nested dictionary whose
structure, once full, will be:

Iteration 0:
- Record 1:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
- Record 2:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
  
 [...]

Iteration 1:
- Record 1:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
- Record 2:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
  
 [...]

[...]

In [ ]:
iter_rec_comp_dict = {}

Print info, register data chunks and fill the above dictionary:

In [ ]:
for index in series.iterations:
    iteration = str(index)

    print("\nIteration " + iteration + ":")
    print("==============")

    # Allocate an empty dictionary associated to this iteration
    iter_rec_comp_dict[iteration] = {}

    i = series.iterations[index]

    for key in i.meshes:
        print("Components of record \"" + key + "\":")

        # Allocate an empty dictionary associated to this record. Notice that
        # 'record' is an OpenPMD mesh object, so it's better to use 'key'
        # instead of 'record' as a key in the dictionary ('record' could also be
        # used, but it makes accessing the key clumsy).
        record = i.meshes[key]
        iter_rec_comp_dict[iteration][key] = {}

        # Load each component of each record as a 'data chunk', i.e., an
        # allocated, but STILL INVALID, NumPy array. Later we will flush all
        # chunks (i.e., basically, fill the NumPy arrays) at once: this leads
        # to better I/O performance compared to flushing a large number of
        # small chunks. That's why we bothered creating the nested dictionary:
        # in this way, we can access the valid NumPy arrays for plotting
        # without having to flush each single chunk.
        # *IMPORTANT*: DO NOT access data chunks until flushing has happened!
        for component in record:
            print("    > " + component)  # 'component' is a string
            iter_rec_comp_dict[iteration][key][component] = record[component].load_chunk()  # *INVALID* 3D NumPy array

        print("")

Flush all registered data chunks, which are now **VALID** 3D NumPy arrays:

In [ ]:
series.flush()

Visualize a 2D movie of the mass density on the xz plane:

In [ ]:
# Select the desired record and component to plot
record    = "hydrobasex_rho_patch00_lev00"  # "hydrobasex_eps_patch00_lev00", "hydrobasex_press_patch00_lev00", "hydrobasex_vel_patch00_lev00"
component = "hydrobasex_rho"  # "hydrobasex_eps", "hydrobasex_press", "hydrobasex_rho", "hydrobasex_velx", "hydrobasex_vely", "hydrobasex_velz"

# Set up the axes for the plot and the colorbar
fig    = plt.figure(figsize = [5.0, 2.75])
axplot = fig.add_axes([0.12, 0.14, 0.75, 0.75])
axclb  = fig.add_axes([0.88, 0.14, 0.02, 0.75])

# Set title and labels
axplot.set_title("Rest-mass density", fontsize = 10., fontweight = "bold", color = "midnightblue")
axplot.set_xlabel("x", fontsize = 7.)
axplot.set_ylabel("z", fontsize = 7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))


# Initialize the camera
camera = Camera(fig)

# Print frames
for iteration in iter_rec_comp_dict:
    # Retrieve the 3D array containing the data
    array3D = iter_rec_comp_dict[iteration][record][component]
    
    # Plot on the (x, z) plane at the half-way value of y
    # Notice that the 3D array is stored in (z, y, x) order
    y_index = int(array3D.shape[1]/2)
    x0     = np.linspace(-0.5, 0.5, array3D.shape[2])
    z0     = np.linspace(-0.04, 0.04, array3D.shape[0])
    image   = axplot.pcolormesh(x0, z0, array3D[:, y_index, :],
                                cmap = "magma", vmin = 0.0, vmax = 1.0)
    axplot.set_ylim(ymin=-0.005, ymax=0.005)
    # Set up the colorbar
    axclb.tick_params(labelsize=7.0)
    fig.colorbar(image, cax = axclb, extend = "neither")
    
    # Print the current iteration
    axplot.text(0.18, 0.42, "Iteration " + iteration,
             fontsize = 8., fontweight = "bold", color = "white")

    # Take a snapshot of the figure at this iteration (needed later for the animation)
    camera.snap()

    plt.close(fig)

In [ ]:
from IPython.display import HTML
animation = camera.animate()
HTML(animation.to_html5_video())

You may (or )delete the simulation directory via the following:


In [ ]:
#%cd ~/
#%rm -r ./simulations/Balsara1_shocktube_xdir_WENOZ

Delete also the minmod simulation

In [ ]:
#%cd ~/
#%rm -r ./simulations/Balsara1_shocktube_xdir_MINMOD